# InterDecagon 

In [1]:
!wget https://snap.stanford.edu/biodata/datasets/10016/files/ChG-InterDecagon_targets.csv.gz

--2025-05-14 19:40:43--  https://snap.stanford.edu/biodata/datasets/10016/files/ChG-InterDecagon_targets.csv.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 441916 (432K) [application/x-gzip]
Saving to: ‘ChG-InterDecagon_targets.csv.gz’

ChG-InterDecagon_ta 100%[===================>] 431.56K   157KB/s    in 2.8s    

2025-05-14 19:40:47 (157 KB/s) - ‘ChG-InterDecagon_targets.csv.gz’ saved [441916/441916]



In [3]:
pwd

'/storage/savi/saveenas/Projects/Magnet/Dataset/Updated_Magnet_DB/BioSNAP'

In [5]:
!gunzip ChG-InterDecagon_targets.csv.gz

In [15]:
# Read as TSV (tab-separated file)
EE = pd.read_csv("ChG-Miner_miner-chem-gene.tsv", sep='\t')
EE

,#Drug,Gene
0,DB00357,P05108
1,DB02721,P00325
2,DB00773,P23219
3,DB07138,Q16539
4,DB08136,P24941
...,...,...
15134,DB01215,P47870
15135,DB06089,P51787
15136,DB01614,P21728
15137,DB00582,P08684


In [16]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

drug_ids = EE['#Drug'].unique().tolist()

molecule = new_client.molecule
results = []
for dbid in drug_ids:
    compounds = molecule.filter(drugbank__contains=dbid)
    for c in compounds:
        results.append({
            "DrugBank_ID": dbid,
            "ChEMBL_ID": c["molecule_chembl_id"],
            "SMILES": c["molecule_structures"]["canonical_smiles"] if c["molecule_structures"] else None,
            "Molecule_Name": c["pref_name"]
        })

df_results = pd.DataFrame(results)
df_results

KeyboardInterrupt: 

In [18]:
from mygene import MyGeneInfo
from tqdm import tqdm
import pandas as pd

mg = MyGeneInfo()
genes = EE['Gene'].unique().tolist()

results = []

# tqdm progress bar without batching
for gene in tqdm(genes, desc="Querying genes"):
    out = mg.querymany([gene], scopes='uniprot', fields='symbol,name', species='all')
    results.extend(out)

# convert to DataFrame
df_gene = pd.DataFrame(results)[['query', 'symbol', 'name']].rename(columns={'query': 'Gene'})
df_gene


Querying genes: 100%|██████████| 2325/2325 [1:22:18<00:00,  2.12s/it]


,Gene,symbol,name
0,P05108,CYP11A1,cytochrome P450 family 11 subfamily A member 1
1,P00325,ADH1B,"alcohol dehydrogenase 1B (class I), beta polyp..."
2,P23219,PTGS1,prostaglandin-endoperoxide synthase 1
3,Q16539,MAPK14,mitogen-activated protein kinase 14
4,P24941,CDK2,cyclin dependent kinase 2
...,...,...,...
2332,Q9Y2Z4,YARS2,tyrosyl-tRNA synthetase 2
2333,Q16718,NDUFA5,NADH:ubiquinone oxidoreductase subunit A5
2334,Q96RP8,KCNA7,potassium voltage-gated channel subfamily A me...
2335,P41250,GARS1,glycyl-tRNA synthetase 1


In [19]:
df_gene.to_csv("ChGminer_BioSnap_targets.csv" ,index= False)

# ChGMiner

In [6]:
!wget https://snap.stanford.edu/biodata/datasets/10002/files/ChG-Miner_miner-chem-gene.tsv.gz

--2025-05-14 19:45:03--  https://snap.stanford.edu/biodata/datasets/10002/files/ChG-Miner_miner-chem-gene.tsv.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 76098 (74K) [application/x-gzip]
Saving to: ‘ChG-Miner_miner-chem-gene.tsv.gz’

ChG-Miner_miner-che 100%[===================>]  74.31K  81.3KB/s    in 0.9s    

2025-05-14 19:45:06 (81.3 KB/s) - ‘ChG-Miner_miner-chem-gene.tsv.gz’ saved [76098/76098]



In [7]:
!gunzip ChG-Miner_miner-chem-gene.tsv.gz

In [19]:
# Read as TSV (tab-separated file)
CC = pd.read_csv("ChG-InterDecagon_targets.csv", sep='\t')
CC

,# Drug,Gene
0,"CID000060752,3757",NaN
1,"CID006918155,2908",NaN
2,"CID103052762,3359",NaN
3,"CID023668479,1230",NaN
4,"CID000028864,1269",NaN
...,...,...
131029,"CID000092721,3426",NaN
131030,"CID000092721,8858",NaN
131031,"CID000092721,10942",NaN
131032,"CID100115355,3242",NaN


In [20]:
# Split by comma into Drug and Gene
CC[['Drug', 'Gene']] = CC['# Drug'].str.split(',', expand=True)
CC



,# Drug,Gene,Drug
0,"CID000060752,3757",3757,CID000060752
1,"CID006918155,2908",2908,CID006918155
2,"CID103052762,3359",3359,CID103052762
3,"CID023668479,1230",1230,CID023668479
4,"CID000028864,1269",1269,CID000028864
...,...,...,...
131029,"CID000092721,3426",3426,CID000092721
131030,"CID000092721,8858",8858,CID000092721
131031,"CID000092721,10942",10942,CID000092721
131032,"CID100115355,3242",3242,CID100115355


In [21]:
CC.to_csv("ChG_InterDecagon.csv" , index = False)

In [9]:
CC = pd.read_csv("ChG_InterDecagon.csv")
CC

,# Drug,Gene,Drug
0,"CID000060752,3757",3757,CID000060752
1,"CID006918155,2908",2908,CID006918155
2,"CID103052762,3359",3359,CID103052762
3,"CID023668479,1230",1230,CID023668479
4,"CID000028864,1269",1269,CID000028864
...,...,...,...
131029,"CID000092721,3426",3426,CID000092721
131030,"CID000092721,8858",8858,CID000092721
131031,"CID000092721,10942",10942,CID000092721
131032,"CID100115355,3242",3242,CID100115355


In [10]:
from pubchempy import get_compounds

def get_pubchem_info(cid):
    try:
        compound = get_compounds(cid.replace("CID", ""), 'cid')[0]
        return pd.Series({
            'SMILES': compound.canonical_smiles,
            'Ligand_Name': compound.iupac_name
        })
    except:
        return pd.Series({'SMILES': None, 'Ligand_Name': None})

# Apply to unique drugs
unique_drugs = CC['Drug'].unique()
drug_info = pd.DataFrame(unique_drugs, columns=['Drug'])
drug_info = drug_info.join(drug_info['Drug'].apply(get_pubchem_info))

# Merge back into main table
CC = CC.merge(drug_info, on='Drug', how='left')
CC

,# Drug,Gene,Drug,SMILES,Ligand_Name
0,"CID000060752,3757",3757,CID000060752,CCCCCCCN(CC)CCCC(C1=CC=C(C=C1)NS(=O)(=O)C)O.CC...,but-2-enedioic acid;N-[4-[4-[ethyl(heptyl)amin...
1,"CID006918155,2908",2908,CID006918155,CC(C)C(=O)OCC(=O)C12C(CC3C1(CC(C4C3CCC5=CC(=O)...,"[2-[(1S,2S,4R,6R,8S,9S,11S,12S,13R)-6-cyclohex..."
2,"CID103052762,3359",3359,CID103052762,CCCNC(CC1=C(C=CC(=C1)Cl)F)COC(C)(C)C,1-(5-chloro-2-fluorophenyl)-3-[(2-methylpropan...
3,"CID023668479,1230",1230,CID023668479,CC1=NN=C(O1)C(=O)NC(C)(C)C2=NC(=C(C(=O)N2C)[O-...,potassium;4-[(4-fluorophenyl)methylcarbamoyl]-...
4,"CID000028864,1269",1269,CID000028864,CCN(CC)C(=O)NC1CN(C2CC3=CNC4=CC=CC(=C34)C2=C1)C,"3-[(6aR,9S)-7-methyl-6,6a,8,9-tetrahydro-4H-in..."
...,...,...,...,...,...
131029,"CID000092721,3426",3426,CID000092721,CC1CCN(C(C1)C(=O)O)C(=O)C(CCCN=C(N)N)NS(=O)(=O...,"(2R,4R)-1-[(2S)-5-(diaminomethylideneamino)-2-..."
131030,"CID000092721,8858",8858,CID000092721,CC1CCN(C(C1)C(=O)O)C(=O)C(CCCN=C(N)N)NS(=O)(=O...,"(2R,4R)-1-[(2S)-5-(diaminomethylideneamino)-2-..."
131031,"CID000092721,10942",10942,CID000092721,CC1CCN(C(C1)C(=O)O)C(=O)C(CCCN=C(N)N)NS(=O)(=O...,"(2R,4R)-1-[(2S)-5-(diaminomethylideneamino)-2-..."
131032,"CID100115355,3242",3242,CID100115355,CC1=C(C(=CC=C1)N2CCN(CC2)C(=O)C3=CN=C(N=C3)C4C...,"(2-cyclopropylpyrimidin-5-yl)-[4-(2,3-dimethyl..."


In [11]:
from mygene import MyGeneInfo
from tqdm import tqdm
import pandas as pd

# Enable tqdm for pandas
tqdm.pandas()

# Initialize mygene
mg = MyGeneInfo()

# Function to get gene info
def get_gene_info(gene_id):
    try:
        result = mg.getgene(int(gene_id), fields='symbol,name')
        return pd.Series({
            'Gene_Symbol': result.get('symbol'),
            'Protein_Name': result.get('name')
        })
    except:
        return pd.Series({'Gene_Symbol': None, 'Protein_Name': None})

# Apply to unique genes with progress tracking
unique_genes = CC['Gene'].unique()
gene_info = pd.DataFrame(unique_genes, columns=['Gene'])
gene_info = gene_info.join(gene_info['Gene'].progress_apply(get_gene_info))

# Merge annotations back into main DataFrame
CC = CC.merge(gene_info, on='Gene', how='left')

# Preview result
print(CC.head())


100%|██████████| 7795/7795 [2:26:05<00:00,  1.12s/it]  

              # Drug  Gene          Drug  \
0  CID000060752,3757  3757  CID000060752   
1  CID006918155,2908  2908  CID006918155   
2  CID103052762,3359  3359  CID103052762   
3  CID023668479,1230  1230  CID023668479   
4  CID000028864,1269  1269  CID000028864   

                                              SMILES  \
0  CCCCCCCN(CC)CCCC(C1=CC=C(C=C1)NS(=O)(=O)C)O.CC...   
1  CC(C)C(=O)OCC(=O)C12C(CC3C1(CC(C4C3CCC5=CC(=O)...   
2               CCCNC(CC1=C(C=CC(=C1)Cl)F)COC(C)(C)C   
3  CC1=NN=C(O1)C(=O)NC(C)(C)C2=NC(=C(C(=O)N2C)[O-...   
4    CCN(CC)C(=O)NC1CN(C2CC3=CNC4=CC=CC(=C34)C2=C1)C   

                                         Ligand_Name Gene_Symbol  \
0  but-2-enedioic acid;N-[4-[4-[ethyl(heptyl)amin...       KCNH2   
1  [2-[(1S,2S,4R,6R,8S,9S,11S,12S,13R)-6-cyclohex...       NR3C1   
2  1-(5-chloro-2-fluorophenyl)-3-[(2-methylpropan...       HTR3A   
3  potassium;4-[(4-fluorophenyl)methylcarbamoyl]-...        CCR1   
4  3-[(6aR,9S)-7-methyl-6,6a,8,9-tetrahydro-4H-in...      

In [12]:
CC.to_csv("BioSNAP_ChG_InterDecagon.csv" , index = False)

In [2]:
import pandas as pd

CC =pd.read_csv("BioSNAP_ChG_InterDecagon.csv")
CC

,# Drug,Gene,Drug,SMILES,Ligand_Name,Gene_Symbol,Protein_Name
0,"CID000060752,3757",3757,CID000060752,CCCCCCCN(CC)CCCC(C1=CC=C(C=C1)NS(=O)(=O)C)O.CC...,but-2-enedioic acid;N-[4-[4-[ethyl(heptyl)amin...,KCNH2,potassium voltage-gated channel subfamily H me...
1,"CID006918155,2908",2908,CID006918155,CC(C)C(=O)OCC(=O)C12C(CC3C1(CC(C4C3CCC5=CC(=O)...,"[2-[(1S,2S,4R,6R,8S,9S,11S,12S,13R)-6-cyclohex...",NR3C1,nuclear receptor subfamily 3 group C member 1
2,"CID103052762,3359",3359,CID103052762,CCCNC(CC1=C(C=CC(=C1)Cl)F)COC(C)(C)C,1-(5-chloro-2-fluorophenyl)-3-[(2-methylpropan...,HTR3A,5-hydroxytryptamine receptor 3A
3,"CID023668479,1230",1230,CID023668479,CC1=NN=C(O1)C(=O)NC(C)(C)C2=NC(=C(C(=O)N2C)[O-...,potassium;4-[(4-fluorophenyl)methylcarbamoyl]-...,CCR1,C-C motif chemokine receptor 1
4,"CID000028864,1269",1269,CID000028864,CCN(CC)C(=O)NC1CN(C2CC3=CNC4=CC=CC(=C34)C2=C1)C,"3-[(6aR,9S)-7-methyl-6,6a,8,9-tetrahydro-4H-in...",CNR2,cannabinoid receptor 2
...,...,...,...,...,...,...,...
131029,"CID000092721,3426",3426,CID000092721,CC1CCN(C(C1)C(=O)O)C(=O)C(CCCN=C(N)N)NS(=O)(=O...,"(2R,4R)-1-[(2S)-5-(diaminomethylideneamino)-2-...",CFI,complement factor I
131030,"CID000092721,8858",8858,CID000092721,CC1CCN(C(C1)C(=O)O)C(=O)C(CCCN=C(N)N)NS(=O)(=O...,"(2R,4R)-1-[(2S)-5-(diaminomethylideneamino)-2-...",PROZ,"protein Z, vitamin K dependent plasma glycopro..."
131031,"CID000092721,10942",10942,CID000092721,CC1CCN(C(C1)C(=O)O)C(=O)C(CCCN=C(N)N)NS(=O)(=O...,"(2R,4R)-1-[(2S)-5-(diaminomethylideneamino)-2-...",PRSS21,serine protease 21
131032,"CID100115355,3242",3242,CID100115355,CC1=C(C(=CC=C1)N2CCN(CC2)C(=O)C3=CN=C(N=C3)C4C...,"(2-cyclopropylpyrimidin-5-yl)-[4-(2,3-dimethyl...",HPD,4-hydroxyphenylpyruvate dioxygenase


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Recap
from tqdm import tqdm

# Register tqdm with pandas
tqdm.pandas()

def recap_single_attachment_fragments(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    recap_tree = Recap.RecapDecompose(mol)
    all_frags = list(recap_tree.GetLeaves().keys())

    # Filter fragments with exactly 1 attachment point (*)
    filtered = [frag for frag in all_frags if frag.count('*') == 1]
    return filtered

# Apply with progress bar
CC['Fragments'] = CC['SMILES'].progress_apply(recap_single_attachment_fragments)
CC

 34%|███▍      | 44663/131034 [01:10<03:54, 368.79it/s] 

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Recap
from tqdm import tqdm
import swifter  # Enables faster parallel apply

# Register tqdm with pandas
tqdm.pandas()

def recap_single_attachment_fragments(smiles):
    # Skip invalid or very short SMILES early
    if not smiles or len(smiles) < 5:
        return []
    
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return []

        recap_tree = Recap.RecapDecompose(mol)
        all_frags = list(recap_tree.GetLeaves().keys())

        # Filter fragments with exactly 1 attachment point (*)
        filtered = [frag for frag in all_frags if frag.count('*') == 1]
        return filtered
    except:
        # Catch any unexpected RDKit or Recap errors
        return []

# Apply function using swifter (fast parallel apply)
CC['Fragments'] = CC['SMILES'].swifter.apply(recap_single_attachment_fragments)

# Optionally: Drop rows with no fragments
CC = CC[CC['Fragments'].str.len() > 0].reset_index(drop=True)

# Note:
# progress_apply is sequential:
# It applies the function to each row one by one,
# which means over 131,000 calls to Chem.MolFromSmiles and Recap.RecapDecompose.
# For sequential run with progress bar:
# CC['Fragments'] = CC['SMILES'].progress_apply(recap_single_attachment_fragments)
CC


Pandas Apply:   0%|          | 0/131034 [00:00<?, ?it/s]

In [3]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Recap
from tqdm import tqdm
import swifter  # Enables faster parallel apply

# Register tqdm with pandas
tqdm.pandas()

def recap_single_attachment_fragments(smiles):
    # Skip invalid or very short SMILES early
    if not smiles or len(smiles) < 5:
        return []

    try:
        # Log the current SMILES being processed (can comment out later)
        print(f"Processing: {smiles}")
        
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return []

        # Skip large molecules (over 100 heavy atoms)
        if mol.GetNumAtoms() > 100:
            return []

        recap_tree = Recap.RecapDecompose(mol)
        all_frags = list(recap_tree.GetLeaves().keys())

        # Filter fragments with exactly 1 attachment point (*)
        filtered = [frag for frag in all_frags if frag.count('*') == 1]
        return filtered

    except:
        # Catch any unexpected RDKit or Recap errors
        return []

# Apply function using swifter (fast parallel apply)
CC['Fragments'] = CC['SMILES'].swifter.apply(recap_single_attachment_fragments)

# Drop rows with no fragments
CC = CC[CC['Fragments'].str.len() > 0].reset_index(drop=True)

# Note:
# progress_apply is sequential:
# It applies the function to each row one by one,
# which means over 131,000 calls to Chem.MolFromSmiles and Recap.RecapDecompose.
# For sequential run with progress bar:
# CC['Fragments'] = CC['SMILES'].progress_apply(recap_single_attachment_fragments)


Pandas Apply:   0%|          | 0/131034 [00:00<?, ?it/s]

In [5]:
CC.to_csv("RECAP_frag_InterDecagon.csv" , index = False)

In [43]:
CC

,# Drug,Gene,Drug,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments
0,"CID000060752,3757",3757,CID000060752,CCCCCCCN(CC)CCCC(C1=CC=C(C=C1)NS(=O)(=O)C)O.CC...,but-2-enedioic acid;N-[4-[4-[ethyl(heptyl)amin...,KCNH2,potassium voltage-gated channel subfamily H me...,"[*O, *CCCCCCC, *S(C)(=O)=O]"
1,"CID006918155,2908",2908,CID006918155,CC(C)C(=O)OCC(=O)C12C(CC3C1(CC(C4C3CCC5=CC(=O)...,"[2-[(1S,2S,4R,6R,8S,9S,11S,12S,13R)-6-cyclohex...",NR3C1,nuclear receptor subfamily 3 group C member 1,"[*C(=O)C(C)C, *OCC(=O)C12OC(C3CCCCC3)OC1CC1C3C..."
2,"CID103052762,3359",3359,CID103052762,CCCNC(CC1=C(C=CC(=C1)Cl)F)COC(C)(C)C,1-(5-chloro-2-fluorophenyl)-3-[(2-methylpropan...,HTR3A,5-hydroxytryptamine receptor 3A,"[*C(C)(C)C, *CC(Cc1cc(Cl)ccc1F)NCCC]"
3,"CID023668479,1230",1230,CID023668479,CC1=NN=C(O1)C(=O)NC(C)(C)C2=NC(=C(C(=O)N2C)[O-...,potassium;4-[(4-fluorophenyl)methylcarbamoyl]-...,CCR1,C-C motif chemokine receptor 1,"[*C(=O)c1nnc(C)o1, *NCc1ccc(F)cc1]"
4,"CID000028864,1269",1269,CID000028864,CCN(CC)C(=O)NC1CN(C2CC3=CNC4=CC=CC(=C34)C2=C1)C,"3-[(6aR,9S)-7-methyl-6,6a,8,9-tetrahydro-4H-in...",CNR2,cannabinoid receptor 2,"[*N(CC)CC, *NC1C=C2c3cccc4[nH]cc(c34)CC2N(C)C1]"
...,...,...,...,...,...,...,...,...
78251,"CID000092721,3426",3426,CID000092721,CC1CCN(C(C1)C(=O)O)C(=O)C(CCCN=C(N)N)NS(=O)(=O...,"(2R,4R)-1-[(2S)-5-(diaminomethylideneamino)-2-...",CFI,complement factor I,"[*O, *S(=O)(=O)c1cccc2c1NCC(C)C2]"
78252,"CID000092721,8858",8858,CID000092721,CC1CCN(C(C1)C(=O)O)C(=O)C(CCCN=C(N)N)NS(=O)(=O...,"(2R,4R)-1-[(2S)-5-(diaminomethylideneamino)-2-...",PROZ,"protein Z, vitamin K dependent plasma glycopro...","[*O, *S(=O)(=O)c1cccc2c1NCC(C)C2]"
78253,"CID000092721,10942",10942,CID000092721,CC1CCN(C(C1)C(=O)O)C(=O)C(CCCN=C(N)N)NS(=O)(=O...,"(2R,4R)-1-[(2S)-5-(diaminomethylideneamino)-2-...",PRSS21,serine protease 21,"[*O, *S(=O)(=O)c1cccc2c1NCC(C)C2]"
78254,"CID100115355,3242",3242,CID100115355,CC1=C(C(=CC=C1)N2CCN(CC2)C(=O)C3=CN=C(N=C3)C4C...,"(2-cyclopropylpyrimidin-5-yl)-[4-(2,3-dimethyl...",HPD,4-hydroxyphenylpyruvate dioxygenase,"[*C(=O)c1cnc(C2CC2)nc1, *c1cccc(C)c1C]"


In [45]:
from rdkit import Chem
import pandas as pd
from tqdm import tqdm

tqdm.pandas()

# Step 1: Assign Database_ID to each unique Drug
unique_drugs_cc = CC['Drug'].unique()
cc_drug_map = {drug: f"BioSnap_TD_{i+1}" for i, drug in enumerate(unique_drugs_cc)}
CC['Database_ID'] = CC['Drug'].map(cc_drug_map)

# Step 2: Filter terminal fragments with only one attachment point
def filter_single_star(frag_list):
    if not isinstance(frag_list, list):
        return []
    return [frag for frag in frag_list if frag.count('*') == 1]

CC['selected_terminal_fragments'] = CC['Fragments'].progress_apply(filter_single_star)

# Step 3: Count selected fragments per row
CC['selected_terminal_fragments_count'] = CC['selected_terminal_fragments'].apply(len)

# Step 4: Count atoms in SMILES
def count_atoms(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol.GetNumHeavyAtoms() if mol else 0
    except:
        return 0

CC['SMILES_Atom_Count'] = CC['SMILES'].progress_apply(count_atoms)

# Step 5: Explode to fragment-wise view
CC_frag_df = CC.explode('selected_terminal_fragments').reset_index(drop=True)
CC_frag_df = CC_frag_df.rename(columns={'selected_terminal_fragments': 'Fragment'})
CC_frag_df = CC_frag_df[CC_frag_df['Fragment'].notnull() & (CC_frag_df['Fragment'] != '')]

# Step 6: Count atoms in each fragment
def count_fragment_atoms(frag):
    try:
        mol = Chem.MolFromSmiles(frag.replace('*', ''))
        return mol.GetNumHeavyAtoms() if mol else 0
    except:
        return 0

CC_frag_df['Fragment_Atom_Count'] = CC_frag_df['Fragment'].progress_apply(count_fragment_atoms)

# Step 7: Calculate Target_Percentage
CC_frag_df['Target_Percentage'] = (CC_frag_df['Fragment_Atom_Count'] / CC_frag_df['SMILES_Atom_Count']) * 100

# Step 8: Reorder columns
first_cols = ['Fragment', 'Database_ID', 'selected_terminal_fragments_count', 'SMILES_Atom_Count', 'Fragment_Atom_Count', 'Target_Percentage']
remaining_cols = [col for col in CC_frag_df.columns if col not in first_cols]
CC_frag_df = CC_frag_df[first_cols + remaining_cols]

# Step 9: Optional - Reset index and drop duplicates if needed
CC_frag_df = CC_frag_df.reset_index(drop=True)
CC_frag_df

  0%|          | 0/170681 [00:00<?, ?it/s][20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized atoms: 0 1 2 4 7
[20:11:49] Can't kekulize mol.  Unkekulized

,Fragment,Database_ID,selected_terminal_fragments_count,SMILES_Atom_Count,Fragment_Atom_Count,Target_Percentage,# Drug,Gene,Drug,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments
0,*O,BioSnap_TD_1,3,60,1,1.666667,"CID000060752,3757",3757,CID000060752,CCCCCCCN(CC)CCCC(C1=CC=C(C=C1)NS(=O)(=O)C)O.CC...,but-2-enedioic acid;N-[4-[4-[ethyl(heptyl)amin...,KCNH2,potassium voltage-gated channel subfamily H me...,"[*O, *CCCCCCC, *S(C)(=O)=O]"
1,*CCCCCCC,BioSnap_TD_1,3,60,7,11.666667,"CID000060752,3757",3757,CID000060752,CCCCCCCN(CC)CCCC(C1=CC=C(C=C1)NS(=O)(=O)C)O.CC...,but-2-enedioic acid;N-[4-[4-[ethyl(heptyl)amin...,KCNH2,potassium voltage-gated channel subfamily H me...,"[*O, *CCCCCCC, *S(C)(=O)=O]"
2,*S(C)(=O)=O,BioSnap_TD_1,3,60,4,6.666667,"CID000060752,3757",3757,CID000060752,CCCCCCCN(CC)CCCC(C1=CC=C(C=C1)NS(=O)(=O)C)O.CC...,but-2-enedioic acid;N-[4-[4-[ethyl(heptyl)amin...,KCNH2,potassium voltage-gated channel subfamily H me...,"[*O, *CCCCCCC, *S(C)(=O)=O]"
3,*C(=O)C(C)C,BioSnap_TD_2,3,39,5,12.820513,"CID006918155,2908",2908,CID006918155,CC(C)C(=O)OCC(=O)C12C(CC3C1(CC(C4C3CCC5=CC(=O)...,"[2-[(1S,2S,4R,6R,8S,9S,11S,12S,13R)-6-cyclohex...",NR3C1,nuclear receptor subfamily 3 group C member 1,"[*C(=O)C(C)C, *OCC(=O)C12OC(C3CCCCC3)OC1CC1C3C..."
4,*OCC(=O)C12OC(C3CCCCC3)OC1CC1C3CCC4=CC(=O)C=CC...,BioSnap_TD_2,3,39,34,87.179487,"CID006918155,2908",2908,CID006918155,CC(C)C(=O)OCC(=O)C12C(CC3C1(CC(C4C3CCC5=CC(=O)...,"[2-[(1S,2S,4R,6R,8S,9S,11S,12S,13R)-6-cyclohex...",NR3C1,nuclear receptor subfamily 3 group C member 1,"[*C(=O)C(C)C, *OCC(=O)C12OC(C3CCCCC3)OC1CC1C3C..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170676,*S(=O)(=O)c1cccc2c1NCC(C)C2,BioSnap_TD_1333,2,36,14,38.888889,"CID000092721,10942",10942,CID000092721,CC1CCN(C(C1)C(=O)O)C(=O)C(CCCN=C(N)N)NS(=O)(=O...,"(2R,4R)-1-[(2S)-5-(diaminomethylideneamino)-2-...",PRSS21,serine protease 21,"[*O, *S(=O)(=O)c1cccc2c1NCC(C)C2]"
170677,*C(=O)c1cnc(C2CC2)nc1,BioSnap_TD_1334,2,25,11,44.000000,"CID100115355,3242",3242,CID100115355,CC1=C(C(=CC=C1)N2CCN(CC2)C(=O)C3=CN=C(N=C3)C4C...,"(2-cyclopropylpyrimidin-5-yl)-[4-(2,3-dimethyl...",HPD,4-hydroxyphenylpyruvate dioxygenase,"[*C(=O)c1cnc(C2CC2)nc1, *c1cccc(C)c1C]"
170678,*c1cccc(C)c1C,BioSnap_TD_1334,2,25,8,32.000000,"CID100115355,3242",3242,CID100115355,CC1=C(C(=CC=C1)N2CCN(CC2)C(=O)C3=CN=C(N=C3)C4C...,"(2-cyclopropylpyrimidin-5-yl)-[4-(2,3-dimethyl...",HPD,4-hydroxyphenylpyruvate dioxygenase,"[*C(=O)c1cnc(C2CC2)nc1, *c1cccc(C)c1C]"
170679,*C(=O)c1cnc(C2CC2)nc1,BioSnap_TD_1334,2,25,11,44.000000,"CID100115355,84842",84842,CID100115355,CC1=C(C(=CC=C1)N2CCN(CC2)C(=O)C3=CN=C(N=C3)C4C...,"(2-cyclopropylpyrimidin-5-yl)-[4-(2,3-dimethyl...",HPDL,4-hydroxyphenylpyruvate dioxygenase like,"[*C(=O)c1cnc(C2CC2)nc1, *c1cccc(C)c1C]"


In [46]:
CC_frag_df

,Fragment,Database_ID,selected_terminal_fragments_count,SMILES_Atom_Count,Fragment_Atom_Count,Target_Percentage,# Drug,Gene,Drug,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments
0,*O,BioSnap_TD_1,3,60,1,1.666667,"CID000060752,3757",3757,CID000060752,CCCCCCCN(CC)CCCC(C1=CC=C(C=C1)NS(=O)(=O)C)O.CC...,but-2-enedioic acid;N-[4-[4-[ethyl(heptyl)amin...,KCNH2,potassium voltage-gated channel subfamily H me...,"[*O, *CCCCCCC, *S(C)(=O)=O]"
1,*CCCCCCC,BioSnap_TD_1,3,60,7,11.666667,"CID000060752,3757",3757,CID000060752,CCCCCCCN(CC)CCCC(C1=CC=C(C=C1)NS(=O)(=O)C)O.CC...,but-2-enedioic acid;N-[4-[4-[ethyl(heptyl)amin...,KCNH2,potassium voltage-gated channel subfamily H me...,"[*O, *CCCCCCC, *S(C)(=O)=O]"
2,*S(C)(=O)=O,BioSnap_TD_1,3,60,4,6.666667,"CID000060752,3757",3757,CID000060752,CCCCCCCN(CC)CCCC(C1=CC=C(C=C1)NS(=O)(=O)C)O.CC...,but-2-enedioic acid;N-[4-[4-[ethyl(heptyl)amin...,KCNH2,potassium voltage-gated channel subfamily H me...,"[*O, *CCCCCCC, *S(C)(=O)=O]"
3,*C(=O)C(C)C,BioSnap_TD_2,3,39,5,12.820513,"CID006918155,2908",2908,CID006918155,CC(C)C(=O)OCC(=O)C12C(CC3C1(CC(C4C3CCC5=CC(=O)...,"[2-[(1S,2S,4R,6R,8S,9S,11S,12S,13R)-6-cyclohex...",NR3C1,nuclear receptor subfamily 3 group C member 1,"[*C(=O)C(C)C, *OCC(=O)C12OC(C3CCCCC3)OC1CC1C3C..."
4,*OCC(=O)C12OC(C3CCCCC3)OC1CC1C3CCC4=CC(=O)C=CC...,BioSnap_TD_2,3,39,34,87.179487,"CID006918155,2908",2908,CID006918155,CC(C)C(=O)OCC(=O)C12C(CC3C1(CC(C4C3CCC5=CC(=O)...,"[2-[(1S,2S,4R,6R,8S,9S,11S,12S,13R)-6-cyclohex...",NR3C1,nuclear receptor subfamily 3 group C member 1,"[*C(=O)C(C)C, *OCC(=O)C12OC(C3CCCCC3)OC1CC1C3C..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170676,*S(=O)(=O)c1cccc2c1NCC(C)C2,BioSnap_TD_1333,2,36,14,38.888889,"CID000092721,10942",10942,CID000092721,CC1CCN(C(C1)C(=O)O)C(=O)C(CCCN=C(N)N)NS(=O)(=O...,"(2R,4R)-1-[(2S)-5-(diaminomethylideneamino)-2-...",PRSS21,serine protease 21,"[*O, *S(=O)(=O)c1cccc2c1NCC(C)C2]"
170677,*C(=O)c1cnc(C2CC2)nc1,BioSnap_TD_1334,2,25,11,44.000000,"CID100115355,3242",3242,CID100115355,CC1=C(C(=CC=C1)N2CCN(CC2)C(=O)C3=CN=C(N=C3)C4C...,"(2-cyclopropylpyrimidin-5-yl)-[4-(2,3-dimethyl...",HPD,4-hydroxyphenylpyruvate dioxygenase,"[*C(=O)c1cnc(C2CC2)nc1, *c1cccc(C)c1C]"
170678,*c1cccc(C)c1C,BioSnap_TD_1334,2,25,8,32.000000,"CID100115355,3242",3242,CID100115355,CC1=C(C(=CC=C1)N2CCN(CC2)C(=O)C3=CN=C(N=C3)C4C...,"(2-cyclopropylpyrimidin-5-yl)-[4-(2,3-dimethyl...",HPD,4-hydroxyphenylpyruvate dioxygenase,"[*C(=O)c1cnc(C2CC2)nc1, *c1cccc(C)c1C]"
170679,*C(=O)c1cnc(C2CC2)nc1,BioSnap_TD_1334,2,25,11,44.000000,"CID100115355,84842",84842,CID100115355,CC1=C(C(=CC=C1)N2CCN(CC2)C(=O)C3=CN=C(N=C3)C4C...,"(2-cyclopropylpyrimidin-5-yl)-[4-(2,3-dimethyl...",HPDL,4-hydroxyphenylpyruvate dioxygenase like,"[*C(=O)c1cnc(C2CC2)nc1, *c1cccc(C)c1C]"


In [47]:
# Filter: Keep only rows where Fragment_Atom_Count > 8
CC_frag_df = CC_frag_df[CC_frag_df['Fragment_Atom_Count'] > 8].reset_index(drop=True)

# Save to CSV
CC_frag_df.to_csv("BioSnap_InterDecagon.csv", index=False)


# ChGTargetDecagon

In [8]:
! wget https://snap.stanford.edu/biodata/datasets/10015/files/ChG-TargetDecagon_targets.csv.gz

--2025-05-14 19:46:46--  https://snap.stanford.edu/biodata/datasets/10015/files/ChG-TargetDecagon_targets.csv.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 61647 (60K) [application/x-gzip]
Saving to: ‘ChG-TargetDecagon_targets.csv.gz’

ChG-TargetDecagon_t 100%[===================>]  60.20K  65.7KB/s    in 0.9s    

2025-05-14 19:46:48 (65.7 KB/s) - ‘ChG-TargetDecagon_targets.csv.gz’ saved [61647/61647]



In [9]:
!gunzip ChG-TargetDecagon_targets.csv.gz

In [12]:
import pandas as pd
LL = pd.read_csv("ChG-TargetDecagon_targets.csv")
LL

,# Drug\tGene
CID000003488,1559
CID000003488,8647
CID000077992,3351
CID000077992,3350
CID000077992,3352
...,...
CID000005152,8484
CID000005152,81491
CID000005152,83551
CID000005152,680


In [13]:
import pandas as pd

# Read as TSV (tab-separated file)
LL = pd.read_csv("ChG-TargetDecagon_targets.csv", sep='\t')


# Clean column names if needed
LL.columns = [col.strip().replace('# ', '').replace('\\', '_') for col in LL.columns]

# Rename columns to meaningful names
LL.columns = ['Drug', 'Gene']

LL


,Drug,Gene
0,"CID000003488,1559",NaN
1,"CID000003488,8647",NaN
2,"CID000077992,3351",NaN
3,"CID000077992,3350",NaN
4,"CID000077992,3352",NaN
...,...,...
18685,"CID000005152,8484",NaN
18686,"CID000005152,81491",NaN
18687,"CID000005152,83551",NaN
18688,"CID000005152,680",NaN


In [17]:


# Split by comma into Drug and Gene
LL[['Drug', 'Gene']] = LL['Drug'].str.split(',', expand=True)
LL


,Drug,Gene
0,CID000003488,1559
1,CID000003488,8647
2,CID000077992,3351
3,CID000077992,3350
4,CID000077992,3352
...,...,...
18685,CID000005152,8484
18686,CID000005152,81491
18687,CID000005152,83551
18688,CID000005152,680


In [18]:
LL.to_csv("ChG-TargetDecagon.csv" , index = False)

In [2]:
import pandas as pd
LL = pd.read_csv("ChG-TargetDecagon.csv")
LL

,Drug,Gene
0,CID000003488,1559
1,CID000003488,8647
2,CID000077992,3351
3,CID000077992,3350
4,CID000077992,3352
...,...,...
18685,CID000005152,8484
18686,CID000005152,81491
18687,CID000005152,83551
18688,CID000005152,680


In [3]:
pip install pubchempy


  Preparing metadata (setup.py) ... done
  Created wheel for pubchempy: filename=PubChemPy-1.0.4-py3-none-any.whl size=13819 sha256=9bcc095c731c9bdfbd90d59512a035f123797060cd88ef23facf79bc645696a3
  Stored in directory: /home/saveenas/.cache/pip/wheels/b0/8c/ba/3b00b89931153bf5a4eaa8e73bd1b0319a879cc45175326854
Successfully built pubchempy
Note: you may need to restart the kernel to use updated packages.


In [4]:
from pubchempy import get_compounds

def get_pubchem_info(cid):
    try:
        compound = get_compounds(cid.replace("CID", ""), 'cid')[0]
        return pd.Series({
            'SMILES': compound.canonical_smiles,
            'Ligand_Name': compound.iupac_name
        })
    except:
        return pd.Series({'SMILES': None, 'Ligand_Name': None})

# Apply to unique drugs
unique_drugs = LL['Drug'].unique()
drug_info = pd.DataFrame(unique_drugs, columns=['Drug'])
drug_info = drug_info.join(drug_info['Drug'].apply(get_pubchem_info))

# Merge back into main table
LL = LL.merge(drug_info, on='Drug', how='left')
LL

,Drug,Gene,SMILES,Ligand_Name
0,CID000003488,1559,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...
1,CID000003488,8647,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...
2,CID000077992,3351,CNC1CCC2=C(C1)C3=C(N2)C=CC(=C3)C(=O)N,"(6R)-6-(methylamino)-6,7,8,9-tetrahydro-5H-car..."
3,CID000077992,3350,CNC1CCC2=C(C1)C3=C(N2)C=CC(=C3)C(=O)N,"(6R)-6-(methylamino)-6,7,8,9-tetrahydro-5H-car..."
4,CID000077992,3352,CNC1CCC2=C(C1)C3=C(N2)C=CC(=C3)C(=O)N,"(6R)-6-(methylamino)-6,7,8,9-tetrahydro-5H-car..."
...,...,...,...,...
18685,CID000005152,8484,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...
18686,CID000005152,81491,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...
18687,CID000005152,83551,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...
18688,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...


In [5]:
from mygene import MyGeneInfo

mg = MyGeneInfo()

def get_gene_info(gene_id):
    try:
        result = mg.getgene(int(gene_id), fields='symbol,name')
        return pd.Series({
            'Gene_Symbol': result.get('symbol'),
            'Protein_Name': result.get('name')
        })
    except:
        return pd.Series({'Gene_Symbol': None, 'Protein_Name': None})

# Apply to unique genes
unique_genes = LL['Gene'].unique()
gene_info = pd.DataFrame(unique_genes, columns=['Gene'])
gene_info = gene_info.join(gene_info['Gene'].apply(get_gene_info))

# Merge back into main table
LL = LL.merge(gene_info, on='Gene', how='left')


In [6]:
LL

,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name
0,CID000003488,1559,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,CYP2C9,cytochrome P450 family 2 subfamily C member 9
1,CID000003488,8647,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,ABCB11,ATP binding cassette subfamily B member 11
2,CID000077992,3351,CNC1CCC2=C(C1)C3=C(N2)C=CC(=C3)C(=O)N,"(6R)-6-(methylamino)-6,7,8,9-tetrahydro-5H-car...",HTR1B,5-hydroxytryptamine receptor 1B
3,CID000077992,3350,CNC1CCC2=C(C1)C3=C(N2)C=CC(=C3)C(=O)N,"(6R)-6-(methylamino)-6,7,8,9-tetrahydro-5H-car...",HTR1A,5-hydroxytryptamine receptor 1A
4,CID000077992,3352,CNC1CCC2=C(C1)C3=C(N2)C=CC(=C3)C(=O)N,"(6R)-6-(methylamino)-6,7,8,9-tetrahydro-5H-car...",HTR1D,5-hydroxytryptamine receptor 1D
...,...,...,...,...,...,...
18685,CID000005152,8484,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GALR3,galanin receptor 3
18686,CID000005152,81491,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GPR63,G protein-coupled receptor 63
18687,CID000005152,83551,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,TAAR8,trace amine associated receptor 8
18688,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3


In [7]:
LL.to_csv("Final_BioSnap_ChGTargetDecagon.csv" , index = False)

In [6]:
LL = pd.read_csv("Final_BioSnap_ChGTargetDecagon.csv")
LL

,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name
0,CID000003488,1559,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,CYP2C9,cytochrome P450 family 2 subfamily C member 9
1,CID000003488,8647,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,ABCB11,ATP binding cassette subfamily B member 11
2,CID000077992,3351,CNC1CCC2=C(C1)C3=C(N2)C=CC(=C3)C(=O)N,"(6R)-6-(methylamino)-6,7,8,9-tetrahydro-5H-car...",HTR1B,5-hydroxytryptamine receptor 1B
3,CID000077992,3350,CNC1CCC2=C(C1)C3=C(N2)C=CC(=C3)C(=O)N,"(6R)-6-(methylamino)-6,7,8,9-tetrahydro-5H-car...",HTR1A,5-hydroxytryptamine receptor 1A
4,CID000077992,3352,CNC1CCC2=C(C1)C3=C(N2)C=CC(=C3)C(=O)N,"(6R)-6-(methylamino)-6,7,8,9-tetrahydro-5H-car...",HTR1D,5-hydroxytryptamine receptor 1D
...,...,...,...,...,...,...
18685,CID000005152,8484,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GALR3,galanin receptor 3
18686,CID000005152,81491,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GPR63,G protein-coupled receptor 63
18687,CID000005152,83551,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,TAAR8,trace amine associated receptor 8
18688,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3


In [7]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Recap
from tqdm import tqdm
import swifter  # Enables faster parallel apply

# Register tqdm with pandas
tqdm.pandas()

def recap_single_attachment_fragments(smiles):
    # Skip invalid or very short SMILES early
    if not smiles or len(smiles) < 5:
        return []

    try:
        # Log the current SMILES being processed (can comment out later)
        print(f"Processing: {smiles}")
        
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return []

        # Skip large molecules (over 100 heavy atoms)
        if mol.GetNumAtoms() > 100:
            return []

        recap_tree = Recap.RecapDecompose(mol)
        all_frags = list(recap_tree.GetLeaves().keys())

        # Filter fragments with exactly 1 attachment point (*)
        filtered = [frag for frag in all_frags if frag.count('*') == 1]
        return filtered

    except:
        # Catch any unexpected RDKit or Recap errors
        return []

# Apply function using swifter (fast parallel apply)
LL['Fragments'] = LL['SMILES'].swifter.apply(recap_single_attachment_fragments)

# Drop rows with no fragments
LL = LL[LL['Fragments'].str.len() > 0].reset_index(drop=True)

# Note:
# progress_apply is sequential:
# It applies the function to each row one by one,
# which means over 131,000 calls to Chem.MolFromSmiles and Recap.RecapDecompose.
# For sequential run with progress bar:
# CC['Fragments'] = CC['SMILES'].progress_apply(recap_single_attachment_fragments)


Pandas Apply:   0%|          | 0/18690 [00:00<?, ?it/s]

Processing: COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)NC(=O)NC3CCCCC3
Processing: COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)NC(=O)NC3CCCCC3
Processing: CNC1CCC2=C(C1)C3=C(N2)C=CC(=C3)C(=O)N
Processing: CNC1CCC2=C(C1)C3=C(N2)C=CC(=C3)C(=O)N
Processing: CNC1CCC2=C(C1)C3=C(N2)C=CC(=C3)C(=O)N
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O
Processing: CC(C)(C)N

In [8]:
LL

,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments
0,CID000003488,1559,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,CYP2C9,cytochrome P450 family 2 subfamily C member 9,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]"
1,CID000003488,8647,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,ABCB11,ATP binding cassette subfamily B member 11,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]"
2,CID000002083,1269,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,CNR2,cannabinoid receptor 2,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]"
3,CID000002083,124274,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR139,G protein-coupled receptor 139,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]"
4,CID000002083,2849,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR26,G protein-coupled receptor 26,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]"
...,...,...,...,...,...,...,...
13028,CID000005152,8484,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GALR3,galanin receptor 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]"
13029,CID000005152,81491,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GPR63,G protein-coupled receptor 63,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]"
13030,CID000005152,83551,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,TAAR8,trace amine associated receptor 8,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]"
13031,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]"


In [9]:
LL.to_csv("ChGTargetDecagon.csv", index = False)

In [10]:
from rdkit import Chem
from tqdm import tqdm

# Register tqdm with pandas
tqdm.pandas()

# Helper: count atoms excluding '*'
def get_heavy_atom_count(fragment):
    frag_clean = fragment.replace('*', '')  # Remove wildcard
    mol = Chem.MolFromSmiles(frag_clean)
    if mol:
        return mol.GetNumHeavyAtoms()
    else:
        return 0

# Filter fragments with >8 heavy atoms
def filter_large_terminal_fragments(frag_list):
    if not isinstance(frag_list, list):
        return []
    return [frag for frag in frag_list if get_heavy_atom_count(frag) > 8]

# Apply with progress bar
LL['selected_terminal_fragments'] = LL['Fragments'].progress_apply(filter_large_terminal_fragments)

# Optional: Add count of selected fragments
LL['selected_terminal_fragments_count'] = LL['selected_terminal_fragments'].apply(len)
 

  0%|          | 0/13033 [00:00<?, ?it/s][18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 2 11 12 13 14 15 16
[18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 2 11 12 13 14 15 16
[18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 2 11 12 13 14 15 16
[18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 2 11 12 13 14 15 16
[18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 2 11 12 13 14 15 16
[18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 2 11 12 13 14 15 16
[18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 2 11 12 13 14 15 16
[18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 6 7 9
[18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 6 7 9
[18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 6 7 9
[18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 6 7 9
[18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 6 7 9
[18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 6 7 9
[18:17:08] Can't kekulize mol.  Unkekulized atoms: 0 1 6 7 9
[18:17:

In [11]:
LL

,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments,selected_terminal_fragments,selected_terminal_fragments_count
0,CID000003488,1559,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,CYP2C9,cytochrome P450 family 2 subfamily C member 9,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",[*C(=O)c1cc(Cl)ccc1OC],1
1,CID000003488,8647,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,ABCB11,ATP binding cassette subfamily B member 11,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",[*C(=O)c1cc(Cl)ccc1OC],1
2,CID000002083,1269,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,CNR2,cannabinoid receptor 2,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",[*CC(O)c1ccc(O)c(CO)c1],1
3,CID000002083,124274,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR139,G protein-coupled receptor 139,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",[*CC(O)c1ccc(O)c(CO)c1],1
4,CID000002083,2849,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR26,G protein-coupled receptor 26,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",[*CC(O)c1ccc(O)c(CO)c1],1
...,...,...,...,...,...,...,...,...,...
13028,CID000005152,8484,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GALR3,galanin receptor 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2
13029,CID000005152,81491,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GPR63,G protein-coupled receptor 63,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2
13030,CID000005152,83551,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,TAAR8,trace amine associated receptor 8,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2
13031,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2


In [13]:
from rdkit import Chem
from tqdm import tqdm

# Register tqdm for pandas
tqdm.pandas()

# 1. Atom count for full SMILES (column 9)
def count_atoms_in_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol.GetNumHeavyAtoms() if mol else 0
    except:
        return 0

LL['SMILES_Atom_Count'] = LL['SMILES'].progress_apply(count_atoms_in_smiles)

# 2. Atom count for each fragment in selected_terminal_fragments (excluding '*')
def count_atoms_in_fragments(fragments):
    atom_counts = []
    if not isinstance(fragments, list):
        return atom_counts
    for frag in fragments:
        try:
            mol = Chem.MolFromSmiles(frag.replace('*', ''))
            if mol:
                atom_counts.append(mol.GetNumHeavyAtoms())
        except:
            continue
    return atom_counts

# Apply and create new column with atom counts for each fragment
LL['selected_terminal_fragments_atom_counts'] = LL['selected_terminal_fragments'].progress_apply(count_atoms_in_fragments)
LL

100%|██████████| 13033/13033 [00:01<00:00, 11973.14it/s]


,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments,selected_terminal_fragments,selected_terminal_fragments_count,SMILES_Atom_Count,selected_terminal_fragments_atom_counts
0,CID000003488,1559,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,CYP2C9,cytochrome P450 family 2 subfamily C member 9,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",[*C(=O)c1cc(Cl)ccc1OC],1,33,[11]
1,CID000003488,8647,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,ABCB11,ATP binding cassette subfamily B member 11,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",[*C(=O)c1cc(Cl)ccc1OC],1,33,[11]
2,CID000002083,1269,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,CNR2,cannabinoid receptor 2,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",[*CC(O)c1ccc(O)c(CO)c1],1,17,[12]
3,CID000002083,124274,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR139,G protein-coupled receptor 139,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",[*CC(O)c1ccc(O)c(CO)c1],1,17,[12]
4,CID000002083,2849,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR26,G protein-coupled receptor 26,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",[*CC(O)c1ccc(O)c(CO)c1],1,17,[12]
...,...,...,...,...,...,...,...,...,...,...,...
13028,CID000005152,8484,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GALR3,galanin receptor 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2,30,"[12, 10]"
13029,CID000005152,81491,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GPR63,G protein-coupled receptor 63,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2,30,"[12, 10]"
13030,CID000005152,83551,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,TAAR8,trace amine associated receptor 8,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2,30,"[12, 10]"
13031,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2,30,"[12, 10]"


In [14]:
from rdkit import Chem
from rdkit.Chem import Recap
from tqdm import tqdm
import swifter

# Register tqdm with pandas
tqdm.pandas()

def recap_single_attachment_fragments(smiles):
    # Skip invalid or very short SMILES
    if not smiles or len(smiles) < 5:
        return []

    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None or mol.GetNumAtoms() > 100:
            return []

        # RECAP decomposition
        recap_tree = Recap.RecapDecompose(mol)
        all_leaves = list(recap_tree.GetLeaves().keys())

        # Keep only fragments with exactly one attachment point (*)
        filtered = [frag for frag in all_leaves if frag.count('*') == 1]
        return filtered

    except:
        return []

LL['Fragments_1'] = LL['SMILES'].swifter.apply(recap_single_attachment_fragments)



Pandas Apply:   0%|          | 0/13033 [00:00<?, ?it/s]

In [15]:
LL

,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments,selected_terminal_fragments,selected_terminal_fragments_count,SMILES_Atom_Count,selected_terminal_fragments_atom_counts,Fragments_1
0,CID000003488,1559,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,CYP2C9,cytochrome P450 family 2 subfamily C member 9,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",[*C(=O)c1cc(Cl)ccc1OC],1,33,[11],"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]"
1,CID000003488,8647,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,ABCB11,ATP binding cassette subfamily B member 11,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",[*C(=O)c1cc(Cl)ccc1OC],1,33,[11],"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]"
2,CID000002083,1269,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,CNR2,cannabinoid receptor 2,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",[*CC(O)c1ccc(O)c(CO)c1],1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]"
3,CID000002083,124274,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR139,G protein-coupled receptor 139,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",[*CC(O)c1ccc(O)c(CO)c1],1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]"
4,CID000002083,2849,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR26,G protein-coupled receptor 26,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",[*CC(O)c1ccc(O)c(CO)c1],1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]"
...,...,...,...,...,...,...,...,...,...,...,...,...
13028,CID000005152,8484,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GALR3,galanin receptor 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]"
13029,CID000005152,81491,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GPR63,G protein-coupled receptor 63,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]"
13030,CID000005152,83551,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,TAAR8,trace amine associated receptor 8,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]"
13031,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]"


In [18]:
LL["Gene"].nunique()

987

In [17]:
LL["SMILES"].nunique()

212

In [16]:
LL["Drug"].nunique()

212

In [19]:
# Step 1: Get unique Drug list
unique_drugs = LL['Drug'].unique()

# Step 2: Create a mapping with BioSnap_TD_X format
drug_id_map = {drug: f"BioSnap_TD_{i+1}" for i, drug in enumerate(unique_drugs)}

# Step 3: Map to LL and create new column
LL['Database_ID'] = LL['Drug'].map(drug_id_map)
LL

,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments,selected_terminal_fragments,selected_terminal_fragments_count,SMILES_Atom_Count,selected_terminal_fragments_atom_counts,Fragments_1,Database_ID
0,CID000003488,1559,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,CYP2C9,cytochrome P450 family 2 subfamily C member 9,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",[*C(=O)c1cc(Cl)ccc1OC],1,33,[11],"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",BioSnap_TD_1
1,CID000003488,8647,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,ABCB11,ATP binding cassette subfamily B member 11,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",[*C(=O)c1cc(Cl)ccc1OC],1,33,[11],"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",BioSnap_TD_1
2,CID000002083,1269,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,CNR2,cannabinoid receptor 2,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",[*CC(O)c1ccc(O)c(CO)c1],1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2
3,CID000002083,124274,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR139,G protein-coupled receptor 139,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",[*CC(O)c1ccc(O)c(CO)c1],1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2
4,CID000002083,2849,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR26,G protein-coupled receptor 26,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",[*CC(O)c1ccc(O)c(CO)c1],1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13028,CID000005152,8484,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GALR3,galanin receptor 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212
13029,CID000005152,81491,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,GPR63,G protein-coupled receptor 63,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212
13030,CID000005152,83551,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,TAAR8,trace amine associated receptor 8,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212
13031,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212


In [20]:
# Step 1: Explode the DataFrame by `selected_terminal_fragments`
LL_frag_df = LL.explode('selected_terminal_fragments').reset_index(drop=True)

# Step 2: Rename the fragment column for clarity
LL_frag_df = LL_frag_df.rename(columns={'selected_terminal_fragments': 'Fragment'})

# Step 3: Drop rows where Fragment is NaN or empty string
LL_frag_df = LL_frag_df[LL_frag_df['Fragment'].notnull()]
LL_frag_df = LL_frag_df[LL_frag_df['Fragment'] != ""]

# Optional: Reset index cleanly
LL_frag_df = LL_frag_df.reset_index(drop=True)
LL_frag_df

,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments,Fragment,selected_terminal_fragments_count,SMILES_Atom_Count,selected_terminal_fragments_atom_counts,Fragments_1,Database_ID
0,CID000003488,1559,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,CYP2C9,cytochrome P450 family 2 subfamily C member 9,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",*C(=O)c1cc(Cl)ccc1OC,1,33,[11],"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",BioSnap_TD_1
1,CID000003488,8647,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,ABCB11,ATP binding cassette subfamily B member 11,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",*C(=O)c1cc(Cl)ccc1OC,1,33,[11],"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",BioSnap_TD_1
2,CID000002083,1269,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,CNR2,cannabinoid receptor 2,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2
3,CID000002083,124274,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR139,G protein-coupled receptor 139,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2
4,CID000002083,2849,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR26,G protein-coupled receptor 26,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14008,CID000005152,83551,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,TAAR8,trace amine associated receptor 8,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",*CCCCc1ccccc1,2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212
14009,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",*CC(O)c1ccc(O)c(CO)c1,2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212
14010,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",*CCCCc1ccccc1,2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212
14011,CID000005152,11255,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,HRH3,histamine receptor H3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",*CC(O)c1ccc(O)c(CO)c1,2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212


In [22]:
# Step 1: Identify list-type columns
list_cols = [col for col in LL_frag_df.columns if LL_frag_df[col].apply(lambda x: isinstance(x, list)).any()]

# Step 2: Keep only hashable (non-list) columns for duplicate removal
hashable_cols = [col for col in LL_frag_df.columns if col not in list_cols]

# Step 3: Drop duplicates based on hashable columns only
LL_frag_df = LL_frag_df.drop_duplicates(subset=hashable_cols).reset_index(drop=True)
LL_frag_df

,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments,Fragment,selected_terminal_fragments_count,SMILES_Atom_Count,selected_terminal_fragments_atom_counts,Fragments_1,Database_ID
0,CID000003488,1559,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,CYP2C9,cytochrome P450 family 2 subfamily C member 9,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",*C(=O)c1cc(Cl)ccc1OC,1,33,[11],"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",BioSnap_TD_1
1,CID000003488,8647,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,ABCB11,ATP binding cassette subfamily B member 11,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",*C(=O)c1cc(Cl)ccc1OC,1,33,[11],"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",BioSnap_TD_1
2,CID000002083,1269,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,CNR2,cannabinoid receptor 2,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2
3,CID000002083,124274,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR139,G protein-coupled receptor 139,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2
4,CID000002083,2849,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR26,G protein-coupled receptor 26,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14008,CID000005152,83551,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,TAAR8,trace amine associated receptor 8,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",*CCCCc1ccccc1,2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212
14009,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",*CC(O)c1ccc(O)c(CO)c1,2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212
14010,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",*CCCCc1ccccc1,2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212
14011,CID000005152,11255,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,HRH3,histamine receptor H3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",*CC(O)c1ccc(O)c(CO)c1,2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212


In [23]:
from rdkit import Chem
from tqdm import tqdm

# Register tqdm with pandas
tqdm.pandas()

# Step 1: Compute SMILES atom count (if not already present)
def count_atoms_in_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol.GetNumHeavyAtoms() if mol else 0
    except:
        return 0

LL_frag_df['SMILES_Atom_Count'] = LL_frag_df['SMILES'].progress_apply(count_atoms_in_smiles)

# Step 2: Compute Fragment atom count (excluding '*')
def count_atoms_in_fragment(fragment):
    try:
        mol = Chem.MolFromSmiles(fragment.replace('*', ''))
        return mol.GetNumHeavyAtoms() if mol else 0
    except:
        return 0

LL_frag_df['Fragment_Atom_Count'] = LL_frag_df['Fragment'].progress_apply(count_atoms_in_fragment)

# Step 3: Calculate Target_Percentage
LL_frag_df['Target_Percentage'] = (LL_frag_df['Fragment_Atom_Count'] / LL_frag_df['SMILES_Atom_Count']) * 100
LL_frag_df

100%|██████████| 14013/14013 [00:01<00:00, 12276.87it/s]


,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments,Fragment,selected_terminal_fragments_count,SMILES_Atom_Count,selected_terminal_fragments_atom_counts,Fragments_1,Database_ID,Fragment_Atom_Count,Target_Percentage
0,CID000003488,1559,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,CYP2C9,cytochrome P450 family 2 subfamily C member 9,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",*C(=O)c1cc(Cl)ccc1OC,1,33,[11],"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",BioSnap_TD_1,11,33.333333
1,CID000003488,8647,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,ABCB11,ATP binding cassette subfamily B member 11,"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",*C(=O)c1cc(Cl)ccc1OC,1,33,[11],"[*NC1CCCCC1, *C(=O)c1cc(Cl)ccc1OC]",BioSnap_TD_1,11,33.333333
2,CID000002083,1269,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,CNR2,cannabinoid receptor 2,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2,12,70.588235
3,CID000002083,124274,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR139,G protein-coupled receptor 139,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2,12,70.588235
4,CID000002083,2849,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR26,G protein-coupled receptor 26,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2,12,70.588235
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14008,CID000005152,83551,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,TAAR8,trace amine associated receptor 8,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",*CCCCc1ccccc1,2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212,10,33.333333
14009,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",*CC(O)c1ccc(O)c(CO)c1,2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212,12,40.000000
14010,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",*CCCCc1ccccc1,2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212,10,33.333333
14011,CID000005152,11255,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,HRH3,histamine receptor H3,"[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",*CC(O)c1ccc(O)c(CO)c1,2,30,"[12, 10]","[*CC(O)c1ccc(O)c(CO)c1, *CCCCc1ccccc1]",BioSnap_TD_212,12,40.000000


In [40]:
LL_frag_df.to_csv("BioSNAP_ChGTargetDecagon.csv" , index = False)

In [30]:
pwd

'/storage/savi/saveenas/Projects/Magnet/Dataset/Updated_Magnet_DB/BioSNAP'

In [26]:
AA = pd.read_csv("/storage/savi/saveenas/Projects/Magnet/Dataset/Updated_Magnet_DB/BioSNAP/BioSNAP_ChGTargetDecagon.csv")
AA 


,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments,Fragment,selected_terminal_fragments_count,SMILES_Atom_Count,selected_terminal_fragments_atom_counts,Fragments_1,Database_ID,Fragment_Atom_Count,Target_Percentage
0,CID000003488,1559,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,CYP2C9,cytochrome P450 family 2 subfamily C member 9,"['*NC1CCCCC1', '*C(=O)c1cc(Cl)ccc1OC']",*C(=O)c1cc(Cl)ccc1OC,1,33,[11],"['*NC1CCCCC1', '*C(=O)c1cc(Cl)ccc1OC']",BioSnap_TD_1,11,33.333333
1,CID000003488,8647,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,ABCB11,ATP binding cassette subfamily B member 11,"['*NC1CCCCC1', '*C(=O)c1cc(Cl)ccc1OC']",*C(=O)c1cc(Cl)ccc1OC,1,33,[11],"['*NC1CCCCC1', '*C(=O)c1cc(Cl)ccc1OC']",BioSnap_TD_1,11,33.333333
2,CID000002083,1269,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,CNR2,cannabinoid receptor 2,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235
3,CID000002083,124274,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR139,G protein-coupled receptor 139,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235
4,CID000002083,2849,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR26,G protein-coupled receptor 26,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14008,CID000005152,83551,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,TAAR8,trace amine associated receptor 8,"['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",*CCCCc1ccccc1,2,30,"[12, 10]","['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",BioSnap_TD_212,10,33.333333
14009,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3,"['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",*CC(O)c1ccc(O)c(CO)c1,2,30,"[12, 10]","['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",BioSnap_TD_212,12,40.000000
14010,CID000005152,680,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,BRS3,bombesin receptor subtype 3,"['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",*CCCCc1ccccc1,2,30,"[12, 10]","['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",BioSnap_TD_212,10,33.333333
14011,CID000005152,11255,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,HRH3,histamine receptor H3,"['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",*CC(O)c1ccc(O)c(CO)c1,2,30,"[12, 10]","['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",BioSnap_TD_212,12,40.000000


In [21]:
AA = AA[['Drug', 'SMILES', 'Ligand_Name' , 'Fragments' ,'Fragment']]
AA

,Drug,SMILES,Ligand_Name,Fragments,Fragment
0,CID000003488,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,"['*NC1CCCCC1', '*C(=O)c1cc(Cl)ccc1OC']",*C(=O)c1cc(Cl)ccc1OC
1,CID000003488,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,"['*NC1CCCCC1', '*C(=O)c1cc(Cl)ccc1OC']",*C(=O)c1cc(Cl)ccc1OC
2,CID000002083,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1
3,CID000002083,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1
4,CID000002083,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1
...,...,...,...,...,...
14008,CID000005152,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,"['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",*CCCCc1ccccc1
14009,CID000005152,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,"['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",*CC(O)c1ccc(O)c(CO)c1
14010,CID000005152,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,"['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",*CCCCc1ccccc1
14011,CID000005152,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,"['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",*CC(O)c1ccc(O)c(CO)c1


In [22]:
AA = AA[['Drug', 'SMILES', 'Ligand_Name','Fragments' ,'Fragment']].drop_duplicates().reset_index(drop=True)
AA

,Drug,SMILES,Ligand_Name,Fragments,Fragment
0,CID000003488,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,"['*NC1CCCCC1', '*C(=O)c1cc(Cl)ccc1OC']",*C(=O)c1cc(Cl)ccc1OC
1,CID000002083,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1
2,CID000003658,C1CN(CCN1CCOCCO)C(C2=CC=CC=C2)C3=CC=C(C=C3)Cl,2-[2-[4-[(4-chlorophenyl)-phenylmethyl]piperaz...,"['*CCOCCO', '*CCO', '*C(c1ccccc1)c1ccc(Cl)cc1']",*C(c1ccccc1)c1ccc(Cl)cc1
3,CID000004932,CCCNCC(COC1=CC=CC=C1C(=O)CCC2=CC=CC=C2)O,1-[2-[2-hydroxy-3-(propylamino)propoxy]phenyl]...,"['*CC(O)CNCCC', '*c1ccccc1C(=O)CCc1ccccc1']",*c1ccccc1C(=O)CCc1ccccc1
4,CID000003954,CN(C)C(=O)C(CC[NH+]1CCC(CC1)(C2=CC=C(C=C2)Cl)O...,4-[4-(4-chlorophenyl)-4-hydroxypiperidin-1-ium...,"['*N(C)C', '*C(=O)C(CC[NH+]1CCC(O)(c2ccc(Cl)cc...",*C(=O)C(CC[NH+]1CCC(O)(c2ccc(Cl)cc2)CC1)(c1ccc...
...,...,...,...,...,...
184,CID000005073,CC1=C(C(=O)N2CCCCC2=N1)CCN3CCC(CC3)C4=NOC5=C4C...,"3-[2-[4-(6-fluoro-1,2-benzoxazol-3-yl)piperidi...","['*CCc1c(C)nc2n(c1=O)CCCC2', '*N1CCC(c2noc3cc(...",*N1CCC(c2noc3cc(F)ccc23)CC1
185,CID000004748,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,"['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",*CCCN1c2ccccc2Sc2ccc(Cl)cc21
186,CID000004748,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,"['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",*N1c2ccccc2Sc2ccc(Cl)cc21
187,CID000005152,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,"['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",*CC(O)c1ccc(O)c(CO)c1


In [23]:
AA.to_csv("BIOSNAP_chemcial.csv" , index = False)

In [ ]:
AA

In [41]:
LL_frag_high_coverage = LL_frag_df[LL_frag_df['Target_Percentage'] >= 50].reset_index(drop=True)
LL_frag_high_coverage

,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments,Fragment,selected_terminal_fragments_count,SMILES_Atom_Count,selected_terminal_fragments_atom_counts,Fragments_1,Database_ID,Fragment_Atom_Count,Target_Percentage
0,CID000002083,1269,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,CNR2,cannabinoid receptor 2,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2,12,70.588235
1,CID000002083,124274,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR139,G protein-coupled receptor 139,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2,12,70.588235
2,CID000002083,2849,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR26,G protein-coupled receptor 26,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2,12,70.588235
3,CID000002083,2847,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,MCHR1,melanin concentrating hormone receptor 1,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2,12,70.588235
4,CID000002083,2844,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR21,G protein-coupled receptor 21,"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"[*C(C)(C)C, *CC(O)c1ccc(O)c(CO)c1]",BioSnap_TD_2,12,70.588235
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8735,CID000004748,83551,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,TAAR8,trace amine associated receptor 8,"[*CCO, *CCCN1c2ccccc2Sc2ccc(Cl)cc21, *N1c2cccc...",*N1c2ccccc2Sc2ccc(Cl)cc21,2,27,"[18, 15]","[*CCO, *CCCN1c2ccccc2Sc2ccc(Cl)cc21, *N1c2cccc...",BioSnap_TD_211,15,55.555556
8736,CID000004748,680,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,BRS3,bombesin receptor subtype 3,"[*CCO, *CCCN1c2ccccc2Sc2ccc(Cl)cc21, *N1c2cccc...",*CCCN1c2ccccc2Sc2ccc(Cl)cc21,2,27,"[18, 15]","[*CCO, *CCCN1c2ccccc2Sc2ccc(Cl)cc21, *N1c2cccc...",BioSnap_TD_211,18,66.666667
8737,CID000004748,680,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,BRS3,bombesin receptor subtype 3,"[*CCO, *CCCN1c2ccccc2Sc2ccc(Cl)cc21, *N1c2cccc...",*N1c2ccccc2Sc2ccc(Cl)cc21,2,27,"[18, 15]","[*CCO, *CCCN1c2ccccc2Sc2ccc(Cl)cc21, *N1c2cccc...",BioSnap_TD_211,15,55.555556
8738,CID000004748,11255,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,HRH3,histamine receptor H3,"[*CCO, *CCCN1c2ccccc2Sc2ccc(Cl)cc21, *N1c2cccc...",*CCCN1c2ccccc2Sc2ccc(Cl)cc21,2,27,"[18, 15]","[*CCO, *CCCN1c2ccccc2Sc2ccc(Cl)cc21, *N1c2cccc...",BioSnap_TD_211,18,66.666667


In [38]:
LL_frag_high_coverage["Drug"].nunique()

152

In [27]:
LL_frag_high_coverage["Gene_Symbol"].nunique()

365

In [42]:
LL_frag_high_coverage.to_csv("BioSNAP_ChGTargetDecagon_high_coverage.csv" , index = False)

In [1]:
import pandas as pd
LL_frag_high_coverage = pd.read_csv("BioSNAP_ChGTargetDecagon_high_coverage.csv")
LL_frag_high_coverage

,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments,Fragment,selected_terminal_fragments_count,SMILES_Atom_Count,selected_terminal_fragments_atom_counts,Fragments_1,Database_ID,Fragment_Atom_Count,Target_Percentage
0,CID000002083,1269,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,CNR2,cannabinoid receptor 2,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235
1,CID000002083,124274,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR139,G protein-coupled receptor 139,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235
2,CID000002083,2849,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR26,G protein-coupled receptor 26,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235
3,CID000002083,2847,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,MCHR1,melanin concentrating hormone receptor 1,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235
4,CID000002083,2844,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR21,G protein-coupled receptor 21,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8735,CID000004748,83551,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,TAAR8,trace amine associated receptor 8,"['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",*N1c2ccccc2Sc2ccc(Cl)cc21,2,27,"[18, 15]","['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",BioSnap_TD_211,15,55.555556
8736,CID000004748,680,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,BRS3,bombesin receptor subtype 3,"['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",*CCCN1c2ccccc2Sc2ccc(Cl)cc21,2,27,"[18, 15]","['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",BioSnap_TD_211,18,66.666667
8737,CID000004748,680,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,BRS3,bombesin receptor subtype 3,"['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",*N1c2ccccc2Sc2ccc(Cl)cc21,2,27,"[18, 15]","['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",BioSnap_TD_211,15,55.555556
8738,CID000004748,11255,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,HRH3,histamine receptor H3,"['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",*CCCN1c2ccccc2Sc2ccc(Cl)cc21,2,27,"[18, 15]","['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",BioSnap_TD_211,18,66.666667


In [3]:
LL_frag_high_coverage = LL_frag_high_coverage.drop(columns=["selected_terminal_fragments_atom_counts", "Fragments_1"])
LL_frag_high_coverage

,Drug,Gene,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments,Fragment,selected_terminal_fragments_count,SMILES_Atom_Count,Database_ID,Fragment_Atom_Count,Target_Percentage
0,CID000002083,1269,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,CNR2,cannabinoid receptor 2,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,BioSnap_TD_2,12,70.588235
1,CID000002083,124274,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR139,G protein-coupled receptor 139,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,BioSnap_TD_2,12,70.588235
2,CID000002083,2849,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR26,G protein-coupled receptor 26,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,BioSnap_TD_2,12,70.588235
3,CID000002083,2847,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,MCHR1,melanin concentrating hormone receptor 1,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,BioSnap_TD_2,12,70.588235
4,CID000002083,2844,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR21,G protein-coupled receptor 21,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,BioSnap_TD_2,12,70.588235
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8735,CID000004748,83551,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,TAAR8,trace amine associated receptor 8,"['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",*N1c2ccccc2Sc2ccc(Cl)cc21,2,27,BioSnap_TD_211,15,55.555556
8736,CID000004748,680,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,BRS3,bombesin receptor subtype 3,"['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",*CCCN1c2ccccc2Sc2ccc(Cl)cc21,2,27,BioSnap_TD_211,18,66.666667
8737,CID000004748,680,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,BRS3,bombesin receptor subtype 3,"['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",*N1c2ccccc2Sc2ccc(Cl)cc21,2,27,BioSnap_TD_211,15,55.555556
8738,CID000004748,11255,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,HRH3,histamine receptor H3,"['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",*CCCN1c2ccccc2Sc2ccc(Cl)cc21,2,27,BioSnap_TD_211,18,66.666667


In [3]:
import pandas as pd
import requests
from mygene import MyGeneInfo
from tqdm import tqdm

tqdm.pandas()

# Load your dataframe (replace this with actual DataFrame or CSV load)
df = LL_frag_high_coverage.copy()

# Optional renaming
df = df.rename(columns={"Gene": "Entrez_ID"})  # if 'Gene' is the Entrez column

# 👉 Process only the first 100 molecules
df = df.head(100).copy()

# Initialize MyGeneInfo
mg = MyGeneInfo()

# Step 1: Map Entrez ID to UniProt (Swiss-Prot preferred)
def get_uniprot(entrez_id):
    try:
        result = mg.getgene(int(entrez_id), fields='uniprot')
        if result and 'uniprot' in result:
            if 'Swiss-Prot' in result['uniprot']:
                return result['uniprot']['Swiss-Prot']
            elif 'TrEMBL' in result['uniprot']:
                return result['uniprot']['TrEMBL']
        return None
    except:
        return None

df['UniProt_ID'] = df['Entrez_ID'].progress_apply(get_uniprot)

# Step 2: Retrieve FASTA sequence
def fetch_fasta(uniprot_id):
    if not uniprot_id:
        return None
    if isinstance(uniprot_id, list):  # Use first if multiple
        uniprot_id = uniprot_id[0]
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    try:
        response = requests.get(url, timeout=10)
        return response.text if response.status_code == 200 else None
    except:
        return None

df['FASTA_Sequence'] = df['UniProt_ID'].progress_apply(fetch_fasta)

# Step 3: Export enriched CSV
df.to_csv("LL_frag_high_coverage_with_uniprot_fasta.csv", index=False)

# Step 4: Save all FASTA to one file
with open("LL_frag_high_coverage_targets.fasta", "w") as f:
    for fasta in df['FASTA_Sequence'].dropna():
        f.write(fasta.strip() + "\n")

print("✅ Done! Mapped UniProt IDs and FASTA sequences saved.")


100%|██████████| 100/100 [01:23<00:00,  1.20it/s]

✅ Done! Mapped UniProt IDs and FASTA sequences saved.


In [4]:
df

,Drug,Entrez_ID,SMILES,Ligand_Name,Gene_Symbol,Protein_Name,Fragments,Fragment,selected_terminal_fragments_count,SMILES_Atom_Count,selected_terminal_fragments_atom_counts,Fragments_1,Database_ID,Fragment_Atom_Count,Target_Percentage,UniProt_ID,FASTA_Sequence
0,CID000002083,1269,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,CNR2,cannabinoid receptor 2,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235,P34972,>sp|P34972|CNR2_HUMAN Cannabinoid receptor 2 O...
1,CID000002083,124274,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR139,G protein-coupled receptor 139,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235,Q6DWJ6,>sp|Q6DWJ6|GP139_HUMAN Probable G-protein coup...
2,CID000002083,2849,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR26,G protein-coupled receptor 26,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235,Q8NDV2,>sp|Q8NDV2|GPR26_HUMAN G-protein coupled recep...
3,CID000002083,2847,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,MCHR1,melanin concentrating hormone receptor 1,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235,Q99705,>sp|Q99705|MCHR1_HUMAN Melanin-concentrating h...
4,CID000002083,2844,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR21,G protein-coupled receptor 21,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235,Q99679,>sp|Q99679|GPR21_HUMAN Probable G-protein coup...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,CID000002083,8477,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GPR65,G protein-coupled receptor 65,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235,Q8IYL9,>sp|Q8IYL9|PSYR_HUMAN Psychosine receptor OS=H...
96,CID000002083,4157,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,MC1R,melanocortin 1 receptor,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235,Q01726,>sp|Q01726|MSHR_HUMAN Melanocyte-stimulating h...
97,CID000002083,84109,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,QRFPR,pyroglutamylated RFamide peptide receptor,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235,Q96P65,>sp|Q96P65|QRFPR_HUMAN Pyroglutamylated RF-ami...
98,CID000002083,2925,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,GRPR,gastrin releasing peptide receptor,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1,1,17,[12],"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",BioSnap_TD_2,12,70.588235,P30550,>sp|P30550|GRPR_HUMAN Gastrin-releasing peptid...


In [9]:
df = df[['UniProt_ID', 'FASTA_Sequence', 'Entrez_ID', 'Protein_Name']]
df

,UniProt_ID,FASTA_Sequence,Entrez_ID,Protein_Name
0,P34972,>sp|P34972|CNR2_HUMAN Cannabinoid receptor 2 O...,1269,cannabinoid receptor 2
1,Q6DWJ6,>sp|Q6DWJ6|GP139_HUMAN Probable G-protein coup...,124274,G protein-coupled receptor 139
2,Q8NDV2,>sp|Q8NDV2|GPR26_HUMAN G-protein coupled recep...,2849,G protein-coupled receptor 26
3,Q99705,>sp|Q99705|MCHR1_HUMAN Melanin-concentrating h...,2847,melanin concentrating hormone receptor 1
4,Q99679,>sp|Q99679|GPR21_HUMAN Probable G-protein coup...,2844,G protein-coupled receptor 21
...,...,...,...,...
95,Q8IYL9,>sp|Q8IYL9|PSYR_HUMAN Psychosine receptor OS=H...,8477,G protein-coupled receptor 65
96,Q01726,>sp|Q01726|MSHR_HUMAN Melanocyte-stimulating h...,4157,melanocortin 1 receptor
97,Q96P65,>sp|Q96P65|QRFPR_HUMAN Pyroglutamylated RF-ami...,84109,pyroglutamylated RFamide peptide receptor
98,P30550,>sp|P30550|GRPR_HUMAN Gastrin-releasing peptid...,2925,gastrin releasing peptide receptor


In [10]:
df.to_csv("Target_sequence.csv", index = False)

In [11]:
pwd

'/storage/savi/saveenas/Projects/Magnet/Dataset/Updated_Magnet_DB/BioSNAP'

In [8]:
df["Entrez_ID"].nunique()

100

In [5]:
df["FASTA_Sequence"][0]

'>sp|P34972|CNR2_HUMAN Cannabinoid receptor 2 OS=Homo sapiens OX=9606 GN=CNR2 PE=1 SV=1\nMEECWVTEIANGSKDGLDSNPMKDYMILSGPQKTAVAVLCTLLGLLSALENVAVLYLILS\nSHQLRRKPSYLFIGSLAGADFLASVVFACSFVNFHVFHGVDSKAVFLLKIGSVTMTFTAS\nVGSLLLTAIDRYLCLRYPPSYKALLTRGRALVTLGIMWVLSALVSYLPLMGWTCCPRPCS\nELFPLIPNDYLLSWLLFIAFLFSGIIYTYGHVLWKAHQHVASLSGHQDRQVPGMARMRLD\nVRLAKTLGLVLAVLLICWFPVLALMAHSLATTLSDQVKKAFAFCSMLCLINSMVNPVIYA\nLRSGEIRSSAHHCLAHWKKCVRGLGSEAKEEAPRSSVTETEADGKITPWPDSRDLDLSDC\n'

In [ ]:
import pandas as pd
import requests
from mygene import MyGeneInfo
from tqdm import tqdm



# Load your dataframe (replace with actual file if needed)
# Example: df = pd.read_csv("LL_frag_high_coverage.csv")
df = LL_frag_high_coverage.copy()

# Ensure Gene Symbol and Entrez ID are treated correctly
df = df.rename(columns={"Gene": "Entrez_ID"})  # if needed
mg = MyGeneInfo()

# Step 1: Map Entrez ID to UniProt (Swiss-Prot preferred)
def get_uniprot(entrez_id):
    try:
        result = mg.getgene(int(entrez_id), fields='uniprot')
        if result and 'uniprot' in result:
            if 'Swiss-Prot' in result['uniprot']:
                return result['uniprot']['Swiss-Prot']
            elif 'TrEMBL' in result['uniprot']:
                return result['uniprot']['TrEMBL']
        return None
    except:
        return None

tqdm.pandas(desc="🔍 Mapping Entrez to UniProt")
df['UniProt_ID'] = df['Entrez_ID'].progress_apply(get_uniprot)

# Step 2: Retrieve FASTA sequence for each UniProt ID
def fetch_fasta(uniprot_id):
    if not uniprot_id:
        return None
    if isinstance(uniprot_id, list):  # handle list of IDs
        uniprot_id = uniprot_id[0]
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    response = requests.get(url)
    return response.text if response.status_code == 200 else None

tqdm.pandas(desc="🧬 Fetching FASTA")
df['FASTA_Sequence'] = df['UniProt_ID'].progress_apply(fetch_fasta)

# Step 3: Export
df.to_csv("LL_frag_high_coverage_with_uniprot_fasta.csv", index=False)

# Step 4: Save all FASTA to a .fasta file
with open("LL_frag_high_coverage_targets.fasta", "w") as f:
    for fasta in df['FASTA_Sequence'].dropna():
        f.write(fasta.strip() + "\n")

print("✅ Done! Mapped UniProt + FASTA saved.")


🔍 Mapping Entrez to UniProt:   1%|          | 80/8740 [01:51<3:05:15,  1.28s/it]

In [25]:
AA

,Drug,SMILES,Ligand_Name,Fragments,Fragment
0,CID000003488,COC1=C(C=C(C=C1)Cl)C(=O)NCCC2=CC=C(C=C2)S(=O)(...,5-chloro-N-[2-[4-(cyclohexylcarbamoylsulfamoyl...,"['*NC1CCCCC1', '*C(=O)c1cc(Cl)ccc1OC']",*C(=O)c1cc(Cl)ccc1OC
1,CID000002083,CC(C)(C)NCC(C1=CC(=C(C=C1)O)CO)O,4-[2-(tert-butylamino)-1-hydroxyethyl]-2-(hydr...,"['*C(C)(C)C', '*CC(O)c1ccc(O)c(CO)c1']",*CC(O)c1ccc(O)c(CO)c1
2,CID000003658,C1CN(CCN1CCOCCO)C(C2=CC=CC=C2)C3=CC=C(C=C3)Cl,2-[2-[4-[(4-chlorophenyl)-phenylmethyl]piperaz...,"['*CCOCCO', '*CCO', '*C(c1ccccc1)c1ccc(Cl)cc1']",*C(c1ccccc1)c1ccc(Cl)cc1
3,CID000004932,CCCNCC(COC1=CC=CC=C1C(=O)CCC2=CC=CC=C2)O,1-[2-[2-hydroxy-3-(propylamino)propoxy]phenyl]...,"['*CC(O)CNCCC', '*c1ccccc1C(=O)CCc1ccccc1']",*c1ccccc1C(=O)CCc1ccccc1
4,CID000003954,CN(C)C(=O)C(CC[NH+]1CCC(CC1)(C2=CC=C(C=C2)Cl)O...,4-[4-(4-chlorophenyl)-4-hydroxypiperidin-1-ium...,"['*N(C)C', '*C(=O)C(CC[NH+]1CCC(O)(c2ccc(Cl)cc...",*C(=O)C(CC[NH+]1CCC(O)(c2ccc(Cl)cc2)CC1)(c1ccc...
...,...,...,...,...,...
184,CID000005073,CC1=C(C(=O)N2CCCCC2=N1)CCN3CCC(CC3)C4=NOC5=C4C...,"3-[2-[4-(6-fluoro-1,2-benzoxazol-3-yl)piperidi...","['*CCc1c(C)nc2n(c1=O)CCCC2', '*N1CCC(c2noc3cc(...",*N1CCC(c2noc3cc(F)ccc23)CC1
185,CID000004748,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,"['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",*CCCN1c2ccccc2Sc2ccc(Cl)cc21
186,CID000004748,C1CN(CCN1CCCN2C3=CC=CC=C3SC4=C2C=C(C=C4)Cl)CCO,2-[4-[3-(2-chlorophenothiazin-10-yl)propyl]pip...,"['*CCO', '*CCCN1c2ccccc2Sc2ccc(Cl)cc21', '*N1c...",*N1c2ccccc2Sc2ccc(Cl)cc21
187,CID000005152,C1=CC=C(C=C1)CCCCOCCCCCCNCC(C2=CC(=C(C=C2)O)CO)O,2-(hydroxymethyl)-4-[1-hydroxy-2-[6-(4-phenylb...,"['*CC(O)c1ccc(O)c(CO)c1', '*CCCCc1ccccc1']",*CC(O)c1ccc(O)c(CO)c1


In [27]:
import time
import pickle
from collections import defaultdict

# Sample TRIE Node structure
class TrieNode:
    def __init__(self):
        self.children = defaultdict(TrieNode)
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()
    
    def insert(self, word):
        node = self.root
        for char in word:
            node = node.children[char]
        node.is_end = True

# Initialize
trie = Trie()
hash_map = {}

# Make sure 'Fragment', 'Drug', 'Gene', 'Gene_Symbol' columns exist in AA
# Keep only unique rows based on Fragment
AA_unique = AA[['Fragment', 'Drug', 'Gene', 'Gene_Symbol']].drop_duplicates().reset_index(drop=True)

# Build TRIE and hash_map
start = time.time()
for index, row in AA_unique.iterrows():
    frag = row['Fragment']
    identifier = {
        'Drug': row['Drug'],
        'Gene': row['Gene'],
        'Gene_Symbol': row['Gene_Symbol']
    }
    trie.insert((frag + "$")[::-1])  # Insert reverse with $
    hash_map[frag] = identifier
end = time.time()
print("TRIE build time:", end - start)

# Save to .pkl
with open('fragment_trie.pkl', 'wb') as f:
    pickle.dump(trie, f)

with open('fragment_hash_map.pkl', 'wb') as f:
    pickle.dump(hash_map, f)


TRIE build time: 0.5289463996887207


In [28]:
pwd


'/storage/savi/saveenas/Projects/Magnet/Dataset/Updated_Magnet_DB/BioSNAP'

In [29]:
import pickle

# Load TRIE
with open('fragment_trie.pkl', 'rb') as f:
    loaded_trie = pickle.load(f)

# Load Hash Map
with open('fragment_hash_map.pkl', 'rb') as f:
    loaded_hash_map = pickle.load(f)

# 🔍 Check: Number of entries in hash map
print("Total fragments in hash map:", len(loaded_hash_map))

# 🔍 Display first 5 entries of the hash map
for i, (frag, info) in enumerate(loaded_hash_map.items()):
    print(f"Fragment: {frag} -> Info: {info}")
    if i >= 4:  # Only show first 5
        break



Total fragments in hash map: 166
Fragment: *C(=O)c1cc(Cl)ccc1OC -> Info: {'Drug': 'CID000003488', 'Gene': 8647, 'Gene_Symbol': 'ABCB11'}
Fragment: *CC(O)c1ccc(O)c(CO)c1 -> Info: {'Drug': 'CID000005152', 'Gene': 11255, 'Gene_Symbol': 'HRH3'}
Fragment: *C(c1ccccc1)c1ccc(Cl)cc1 -> Info: {'Drug': 'CID000004034', 'Gene': 3269, 'Gene_Symbol': 'HRH1'}
Fragment: *c1ccccc1C(=O)CCc1ccccc1 -> Info: {'Drug': 'CID000004932', 'Gene': 3350, 'Gene_Symbol': 'HTR1A'}
Fragment: *C(=O)C(CC[NH+]1CCC(O)(c2ccc(Cl)cc2)CC1)(c1ccccc1)c1ccccc1 -> Info: {'Drug': 'CID000003954', 'Gene': 11255, 'Gene_Symbol': 'HRH3'}
